# DATA-445: In-Class Assignment 15

Consider the `framingham.csv` data file. The dataset is available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

**Demographic:**
- Sex: male or female (Nominal)
- Age: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

**Behavioral:**
- Current Smoker: whether or not the patient is a current smoker (Nominal)
- Cigs Per Day: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette)

**Medical (history):**
- BP Meds: whether or not the patient was on blood pressure medication (Nominal)
- Prevalent Stroke: whether or not the patient had previously had a stroke (Nominal)
- Prevalent Hyp: whether or not the patient was hypertensive (Nominal)
- Diabetes: whether or not the patient had diabetes (Nominal)

**Medical (current):**
- Tot Chol: total cholesterol level (Continuous)
- Sys BP: systolic blood pressure (Continuous)
- Dia BP: diastolic blood pressure (Continuous)
- BMI: Body Mass Index (Continuous)
- Heart Rate: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values)
- Glucose: glucose level (Continuous)

**Predict variable (desired target):**
- 10 year risk of coronary heart disease CHD (binary: "1" means "Yes", "0" means "No")

In **Python**, answer the following:


## 1. (4 points)

Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [7]:
# Your code here
import pandas as pd

heart = pd.read_csv('framingham.csv')
heart = heart.dropna().reset_index(drop=True)
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## 2. (4 points)

Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` is the target variable. Define the 5-fold cross-validation strategy.

In [8]:
# Your code here
from sklearn.model_selection import StratifiedKFold

X = heart[['age', 'currentSmoker', 'totChol', 'BMI', 'heartRate']]
Y = heart['TenYearCHD']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 3. (8 points)

Train and evaluate an `ExtraTreesClassifier` model (with `n_estimators=500`, and `max_depth=3`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [15]:
# Your code here
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import recall_score

et_md = ExtraTreesClassifier(n_estimators=500, max_depth=3, random_state=42)

et_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = Y.iloc[train_index], Y.iloc[test_index]

    et_md.fit(X_train, y_train)
    y_pred_proba = et_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.1).astype(int)

    score = recall_score(y_test, y_pred)
    et_recalls.append(score)
    print(f'Fold {i+1} Recall: {score}')

average_recall_et = sum(et_recalls) / len(et_recalls)
print(f'Average Recall for Extra Trees Classifier: {average_recall_et}')





Fold 1 Recall: 0.9821428571428571
Fold 2 Recall: 0.9819819819819819
Fold 3 Recall: 1.0
Fold 4 Recall: 1.0
Fold 5 Recall: 0.9732142857142857
Average Recall for Extra Trees Classifier: 0.9874678249678249


## 4. (8 points)

Train and evaluate a `GradientBoostingClassifier` model (with `n_estimators=500` and `max_depth=3`, and `learning_rate=0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [16]:
# Your code here
from sklearn.ensemble import GradientBoostingClassifier

gb_md = GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, random_state=42)

gb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = Y.iloc[train_index], Y.iloc[test_index]

    gb_md.fit(X_train, y_train)
    y_pred_proba = gb_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.1).astype(int)

    score = recall_score(y_test, y_pred)
    gb_recalls.append(score)
    print(f'Fold {i+1} Recall: {score}')

average_recall_gb = sum(gb_recalls) / len(gb_recalls)
print(f'Average Recall for Gradient Boosting Classifier: {average_recall_gb}')

Fold 1 Recall: 0.7053571428571429
Fold 2 Recall: 0.8468468468468469
Fold 3 Recall: 0.918918918918919
Fold 4 Recall: 0.8558558558558559
Fold 5 Recall: 0.8303571428571429
Average Recall for Gradient Boosting Classifier: 0.8314671814671815


## 5. (8 points)

Train and evaluate a `HistGradientBoostingClassifier` model (with `max_iter=500` and `max_depth=3`, and `learning_rate=0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [17]:
# Your code here
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_md = HistGradientBoostingClassifier(max_iter=500, max_depth=3, learning_rate=0.01, random_state=42)

hgb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = Y.iloc[train_index], Y.iloc[test_index]

    hgb_md.fit(X_train, y_train)
    y_pred_proba = hgb_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.1).astype(int)

    score = recall_score(y_test, y_pred)
    hgb_recalls.append(score)
    print(f'Fold {i+1} Recall: {score}')

average_recall_hgb = sum(hgb_recalls) / len(hgb_recalls)
print(f'Average Recall for Hist Gradient Boosting Classifier: {average_recall_hgb}')

c:\Users\15154\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\15154\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\15154\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\15154\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\15154\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Fold 1 Recall: 0.75
Fold 2 Recall: 0.8468468468468469
Fold 3 Recall: 0.9279279279279279
Fold 4 Recall: 0.8918918918918919
Fold 5 Recall: 0.8303571428571429
Average Recall for Hist Gradient Boosting Classifier: 0.849404761904762


## 6. (8 points)

Train and evaluate a `LGBMClassifier` model (with `max_iter=500` and `max_depth=3`, and `learning_rate=0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [23]:
from lightgbm import LGBMClassifier

lgb_md = LGBMClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.01,
    random_state=42,
    force_row_wise=True,
    verbosity=-1
)

lgb_recalls = []

for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = Y.iloc[train_index], Y.iloc[test_index]

    lgb_md.fit(X_train, y_train)
    y_pred_proba = lgb_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.1).astype(int)

    score = recall_score(y_test, y_pred)
    lgb_recalls.append(score)
    print(f"Fold {i+1} Recall: {score:.4f}")

average_recall_lgb = sum(lgb_recalls) / len(lgb_recalls)
print(f"Average Recall for LGBM Classifier: {average_recall_lgb:.4f}")

Fold 1 Recall: 0.7411
Fold 2 Recall: 0.8468
Fold 3 Recall: 0.9279
Fold 4 Recall: 0.8919
Fold 5 Recall: 0.8304
Average Recall for LGBM Classifier: 0.8476


## 7. (8 points)

Using the models from parts 3 to 6 as bases estimators, train and evaluate a `StackingClassifier` over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [25]:
# Your code here
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

log_md = LogisticRegression(max_iter=1000, random_state=42)


stacking_md = StackingClassifier(estimators=[
    ('extra_trees', et_md),
    ('gradient_boosting', gb_md),
    ('hist_gradient_boosting', hgb_md),
    ('lgbm', lgb_md)],
    final_estimator=log_md, cv=5, n_jobs=-1)


stacking_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = Y.iloc[train_index], Y.iloc[test_index]

    stacking_md.fit(X_train, y_train)
    y_pred_proba = stacking_md.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.1).astype(int)

    score = recall_score(y_test, y_pred)
    stacking_recalls.append(score)
    print(f'Fold {i+1} Recall: {score}')

average_recall_stacking = sum(stacking_recalls) / len(stacking_recalls)
print(f'Average Recall for Stacking Classifier: {average_recall_stacking}')

Fold 1 Recall: 0.7857142857142857
Fold 2 Recall: 0.8828828828828829
Fold 3 Recall: 0.972972972972973
Fold 4 Recall: 0.918918918918919
Fold 5 Recall: 0.8660714285714286
Average Recall for Stacking Classifier: 0.8853120978120979


In [26]:
stacking_md

StackingClassifier(cv=5,
                   estimators=[('extra_trees',
                                ExtraTreesClassifier(max_depth=3,
                                                     n_estimators=500,
                                                     random_state=42)),
                               ('gradient_boosting',
                                GradientBoostingClassifier(learning_rate=0.01,
                                                           n_estimators=500,
                                                           random_state=42)),
                               ('hist_gradient_boosting',
                                HistGradientBoostingClassifier(learning_rate=0.01,
                                                               max_depth=3,
                                                               max_iter=500,
                                                               random_state=42)),
                               ('lgbm',
                                LGBMClassifier(force_row_wise=True,
                                               learning_rate=0.01, max_depth=3,
                                               n_estimators=500,
                                               random_state=42,
                                               verbosity=-1))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      random_state=42),
                   n_jobs=-1)

## 8. (3 points)

Using the results from parts 3-7, what model would you use to predict `TenYearCHD` based on the recall?

Based on recall ExtraTreesClassifier wins by a long shot with a recall of "Average Recall for Extra Trees Classifier: 0.9874678249678249"